In [18]:
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import sys

# Ensure openpyxl is available (required for Excel export)
try:
    import openpyxl
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openpyxl'])
    import openpyxl

# Clear any existing handlers to prevent duplicates
root_logger = logging.getLogger()
if root_logger.handlers:
    for handler in root_logger.handlers:
        root_logger.removeHandler(handler)

# Configure logging FIRST before importing DataLoader
logging.basicConfig(level=logging.WARNING, force=True)

# Suppress specific loggers that are verbose
for logger_name in ['__main__', 'snowflake', 'azure', 'urllib3']:
    logging.getLogger(logger_name).setLevel(logging.WARNING)

# Add the project root to the path to import custom modules
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import the DataLoader class
from scripts.data_preparation.load import DataLoader

print("✓ Libraries imported successfully")
print(f"✓ openpyxl {openpyxl.__version__} available")
print("✓ Logging configured (INFO messages suppressed)")


✓ Libraries imported successfully
✓ openpyxl 3.1.5 available
✓ Logging configured (INFO messages suppressed)


In [19]:
# Getting email data from startdate to enddate
startdate = '2026-05-01'
enddate = '2026-05-31' # '2026-05-11'

print(f"Tracking Email History will be Retrieved from {startdate} to {enddate}")

Tracking Email History will be Retrieved from 2026-05-01 to 2026-05-31


In [20]:
# Force reload the DataLoader module to ensure latest code is used
import importlib
import sys

# Remove from cache if present, then reimport fresh
if 'scripts.data_preparation.load' in sys.modules:
    del sys.modules['scripts.data_preparation.load']

from scripts.data_preparation.load import DataLoader
print("✓ DataLoader module loaded fresh")

# Verify the parameter substitution method being used
import inspect
src = inspect.getsource(DataLoader.load_from_snowflake)
if 'query.replace' in src:
    print("✓ Using str.replace() for parameter substitution (correct)")
elif 'query.format' in src:
    print("⚠️  Still using str.format() - reload did not pick up changes!")


✓ DataLoader module loaded fresh
⚠️  Still using str.format() - reload did not pick up changes!


## 1. Load Tracking CheckCall Data from Snowflake

Load the tracking checkcall data from Snowflake using the DataLoader class.

In [ ]:
# Execute the query to get historical load data
print("Querying Snowflake for Tracking checkcall data...")
print(f"Date Range: {startdate} to {enddate}")

loader = DataLoader()

df = loader.load_from_snowflake(
    sql_file_path='../sql/loadTrackingOverprocessing2.sql',
    params={'startdate': startdate, 'enddate': enddate}
)

# Convert datetime columns immediately after load.
# All three are cast to ::TIMESTAMP_NTZ::string in SQL, which strips the timezone
# offset while preserving the Chicago local time value — no utc=True needed.
df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], errors='coerce')
df['OG_APPTOPEN']          = pd.to_datetime(df['OG_APPTOPEN'], errors='coerce')
df['OG_SCHEDDATETIME']     = pd.to_datetime(df['OG_SCHEDDATETIME'], errors='coerce')

print(f"\n✓ Loaded {len(df):,} rows from Snowflake")
print(f"  Date range in data: {df['ENTERED_DATETIME_CST'].min()} to {df['ENTERED_DATETIME_CST'].max()}")
print(f"  Columns: {len(df.columns)}")
df.head()


Querying Snowflake for Tracking checkcall data...
Date Range: 2026-05-01 to 2026-05-31


/home/azureuser/loadTrackingOverProcessing/LoadTrackingOverProcessing/scripts/data_preparation/load.py:169: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, self.snowflake_conn)


In [ ]:
# df[(df['LOAD_NUM']==546748574) & (pd.isnull(df['JOB_FAMILY_DESCRIPTION'])==False)][['LOAD_NUM','EAR2NAME','DESCRIPTION','ENTERED_DATETIME_CST','OG_APPTOPEN','OG_SCHEDDATETIME','UPDATE_USER']]


In [ ]:
# df[(df['LOAD_NUM']==548544546)][['LOAD_NUM','EAR2NAME','DESCRIPTION','ENTERED_DATETIME_CST','OG_APPTOPEN','OG_SCHEDDATETIME','UPDATE_USER']] # & (pd.isnull(df['JOB_FAMILY_DESCRIPTION'])==False)][['LOAD_NUM','EAR2NAME','DESCRIPTION','ENTERED_DATETIME_CST','OG_APPTOPEN','OG_SCHEDDATETIME','UPDATE_USER']]
# # # 546748574

In [ ]:
df[df['LOAD_NUM']==552545690] # 554287901

,LOAD_NUM,CUSTOMERCODE,EAR2ID,EAR2NAME,CHECK_CALL_TYPE,DESCRIPTION,ENTERED_DATETIME_CST,CITY,STATE,COUNTRY,...,TRACKING_METHOD_TYPE,APPT_TYPE,OG_SCHEDDATETIME,OG_APPTOPEN,IS_CROSS_BORDER_RELATED,TRACKING_SLA_PRE_PICK,TRACKING_SLA_IN_TRANSIT,TRACKING_SLA_TOTAL,TRACKING_SLA_TOTAL_SCORE_ACTUAL,TRACKING_SLA_TOTAL_SCORE_POSSIBLE
14009844,552545690,C597656,274.0,The Procter & Gamble Company,CC,Check Call,2026-05-11 11:01:00,DALLAS,TX,US,...,GPS,None,2026-05-11 12:17:00,2026-05-11 13:30:00,0,1.0,1.0,1.0,9.0,9.0
14026962,552545690,C597656,274.0,The Procter & Gamble Company,CC,Check Call,2026-05-11 11:16:00,DUNCANVILLE,TX,US,...,GPS,None,2026-05-11 12:17:00,2026-05-11 13:30:00,0,1.0,1.0,1.0,9.0,9.0
14044616,552545690,C597656,274.0,The Procter & Gamble Company,CC,Check Call,2026-05-11 11:31:00,HUTCHINS,TX,US,...,GPS,None,2026-05-11 12:17:00,2026-05-11 13:30:00,0,1.0,1.0,1.0,9.0,9.0
14061164,552545690,C597656,274.0,The Procter & Gamble Company,CC,Check Call,2026-05-11 11:46:00,WILMER,TX,US,...,GPS,None,2026-05-11 12:17:00,2026-05-11 13:30:00,0,1.0,1.0,1.0,9.0,9.0
14086861,552545690,C597656,274.0,The Procter & Gamble Company,CC,Check Call,2026-05-11 12:06:00,WILMER,TX,US,...,GPS,None,2026-05-11 12:17:00,2026-05-11 13:30:00,0,1.0,1.0,1.0,9.0,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15609544,552545690,C597656,274.0,The Procter & Gamble Company,CC,Check Call,2026-05-12 12:50:00,MERCEDES,TX,US,...,GPS,None,NaT,NaT,0,1.0,1.0,1.0,9.0,9.0
15612443,552545690,C597656,274.0,The Procter & Gamble Company,DD,Depart Drop,2026-05-12 12:52:00,WESLACO,TX,US,...,GPS,None,NaT,NaT,0,1.0,1.0,1.0,9.0,9.0
15627691,552545690,C597656,274.0,The Procter & Gamble Company,CC,Check Call,2026-05-12 13:02:00,SAN BENITO,TX,US,...,GPS,None,NaT,NaT,0,1.0,1.0,1.0,9.0,9.0
15739787,552545690,None,NaN,None,D-Close,W4929838,2026-05-12 14:30:00,WESLACO,TX,United States,...,None,RESCHEDULES SET,NaT,NaT,0,1.0,1.0,1.0,9.0,9.0


In [ ]:
df.columns

Index(['LOAD_NUM', 'CUSTOMERCODE', 'EAR2ID', 'EAR2NAME', 'CHECK_CALL_TYPE',
       'DESCRIPTION', 'ENTERED_DATETIME_CST', 'CITY', 'STATE', 'COUNTRY',
       'LATITUDE', 'LONGITUDE', 'AUTOMATED', 'IS_DIGITAL', 'IS_PREDICTED',
       'UPDATE_USER', 'JOB_FAMILY_DESCRIPTION', 'ROLLUPBRANCHNAME',
       'BRANCHSUBREGION2', 'HUMAN_ENTERED_CHECKCALL_FLAG',
       'TRACKING_IDENTIFIER_CLEAN', 'TRACKING_IDENTIFIER_TYPE',
       'TRACKING_METHOD_TYPE', 'APPT_TYPE', 'OG_SCHEDDATETIME', 'OG_APPTOPEN',
       'IS_CROSS_BORDER_RELATED', 'TRACKING_SLA_PRE_PICK',
       'TRACKING_SLA_IN_TRANSIT', 'TRACKING_SLA_TOTAL',
       'TRACKING_SLA_TOTAL_SCORE_ACTUAL', 'TRACKING_SLA_TOTAL_SCORE_POSSIBLE'],
      dtype='object')

In [ ]:
def get_loads_with_manual_cc_before_pickup(df, early_threshold_hours=4):
    """
    Identifies LOAD_NUMs that have at least one manual check call
    (CHECK_CALL_TYPE = 'CC' and AUTOMATED = False) recorded before
    the original pickup open appointment (OG_APPTOPEN).

    Uses OG_APPTOPEN rather than the ENTERED_DATETIME_CST of the first P-Open
    event, because the appointment time in effect at the time of the check call
    may differ from when the P-Open was eventually recorded (e.g. after reschedules).
    OG_APPTOPEN is the appointment open time that was active when the CC was entered.
    Note: a single load may have multiple different OG_APPTOPEN values across its
    manual CCs as appointments are rescheduled, so it is not included in the output.

    Excludes check calls where UPDATE_USER = 'DATASCIENCE'.
    Only includes check calls where HUMAN_ENTERED_CHECKCALL_FLAG = 1.

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake containing columns:
                           LOAD_NUM, CHECK_CALL_TYPE, AUTOMATED, ENTERED_DATETIME_CST,
                           UPDATE_USER, HUMAN_ENTERED_CHECKCALL_FLAG, OG_APPTOPEN
        early_threshold_hours (int): Hours before OG_APPTOPEN to classify as "very early". Default 4.

    Returns:
        pd.DataFrame: DataFrame of qualifying LOAD_NUMs with supporting details:
                      - LOAD_NUM
                      - manual_cc_before_pickup_count: number of manual CCs before OG_APPTOPEN
                      - earliest_manual_cc: datetime of the first manual CC before OG_APPTOPEN
                      - manual_cc_early_count: manual CCs more than `early_threshold_hours`
                                               before OG_APPTOPEN (very premature touches)
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], errors='coerce')
    df['OG_APPTOPEN'] = pd.to_datetime(df['OG_APPTOPEN'], errors='coerce')

    # Get manual check calls (CC + not automated + not DATASCIENCE user + human entered)
    # Include OG_APPTOPEN — the appointment open time active at the time of each CC
    manual_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] != 'DATASCIENCE') &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1) &
        (df['OG_APPTOPEN'].notna())
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST', 'OG_APPTOPEN']].copy()

    # Keep only manual CCs that occurred BEFORE their row's OG_APPTOPEN
    manual_cc_before = manual_cc[
        manual_cc['ENTERED_DATETIME_CST'] < manual_cc['OG_APPTOPEN']
    ].copy()

    # Flag CCs that are more than `early_threshold_hours` before OG_APPTOPEN
    early_cutoff = pd.Timedelta(hours=early_threshold_hours)
    manual_cc_before['is_very_early'] = (
        manual_cc_before['OG_APPTOPEN'] - manual_cc_before['ENTERED_DATETIME_CST']
    ) > early_cutoff

    # Aggregate per load
    result = (
        manual_cc_before.groupby('LOAD_NUM')
        .agg(
            manual_cc_before_pickup_count=('ENTERED_DATETIME_CST', 'count'),
            earliest_manual_cc=('ENTERED_DATETIME_CST', 'min'),
            manual_cc_early_count=('is_very_early', 'sum')
        )
        .reset_index()
    )

    result['manual_cc_early_count'] = result['manual_cc_early_count'].astype(int)
    result = result.sort_values('manual_cc_before_pickup_count', ascending=False)

    return result


# Run the function
total_loads = df['LOAD_NUM'].nunique()
loads_with_early_manual_cc = get_loads_with_manual_cc_before_pickup(df, early_threshold_hours=4)
qualifying_loads = len(loads_with_early_manual_cc)
total_before_pickup = loads_with_early_manual_cc['manual_cc_before_pickup_count'].sum()
total_very_early = loads_with_early_manual_cc['manual_cc_early_count'].sum()

print(f"Total distinct LOAD_NUMs in dataset:               {total_loads:,}")
print(f"LOAD_NUMs with manual CC before OG_APPTOPEN:       {qualifying_loads:,}")
print(f"Percentage:                                        {qualifying_loads / total_loads * 100:.1f}%")
print(f"Total manual CCs before OG_APPTOPEN:               {total_before_pickup:,}")
print(f"  of which >4 hrs before OG_APPTOPEN:              {total_very_early:,} ({total_very_early / total_before_pickup * 100:.1f}%)")
loads_with_early_manual_cc.head(10)


Total distinct LOAD_NUMs in dataset:               311,364
LOAD_NUMs with manual CC before OG_APPTOPEN:       14,359
Percentage:                                        4.6%
Total manual CCs before OG_APPTOPEN:               15,725
  of which >4 hrs before OG_APPTOPEN:              7,159 (45.5%)


,LOAD_NUM,manual_cc_before_pickup_count,earliest_manual_cc,manual_cc_early_count
14221,555191784,8,2026-05-28 17:43:00,8
7746,553044827,7,2026-05-11 15:42:00,7
12603,554287901,6,2026-05-21 19:54:00,4
51,548544546,6,2026-05-05 17:48:00,6
2111,551828882,5,2026-05-06 17:24:00,2
7069,552902148,4,2026-05-13 07:32:00,4
13107,554475186,4,2026-05-26 10:09:00,3
13087,554469157,4,2026-05-28 10:41:00,1
2649,551986260,4,2026-05-14 07:29:00,2
6478,552817819,4,2026-05-14 09:45:00,1


In [ ]:
def get_early_manual_cc_summary(df, early_threshold_hours=4):
    """
    Summarizes manual check calls that occurred more than `early_threshold_hours`
    before the original pickup open appointment (OG_APPTOPEN) at three aggregated
    levels, plus a raw row-level detail table:
      1. EAR2NAME — to identify which customer segments drive the most early touches
      2. UPDATE_USER + ROLLUPBRANCHNAME — to identify which dispatcher/branch combos
         are entering the most premature manual CCs
      3. UPDATE_USER + ROLLUPBRANCHNAME + EAR2NAME — to identify which user/branch/customer
         segment combos are driving the most premature manual CCs
      4. Raw detail — one row per qualifying manual CC with load/user/customer context

    Uses the same manual CC filter as get_loads_with_manual_cc_before_pickup:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == False
        - UPDATE_USER != 'DATASCIENCE'
        - HUMAN_ENTERED_CHECKCALL_FLAG == 1
        - OG_APPTOPEN is not null
        - ENTERED_DATETIME_CST < OG_APPTOPEN
        - OG_APPTOPEN - ENTERED_DATETIME_CST > early_threshold_hours

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake.
        early_threshold_hours (int): Hours threshold. Default 4.

    Returns:
        tuple:
            - by_ear2 (pd.DataFrame): Aggregated by EAR2NAME
            - by_user_branch (pd.DataFrame): Aggregated by UPDATE_USER + ROLLUPBRANCHNAME
            - by_user_branch_ear2 (pd.DataFrame): Aggregated by UPDATE_USER + ROLLUPBRANCHNAME + EAR2NAME
            - detail (pd.DataFrame): Row-level dump of all qualifying manual CCs with
                                     LOAD_NUM, ENTERED_DATETIME_CST, UPDATE_USER,
                                     CUSTOMERCODE, EAR2NAME, EAR2ID
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], errors='coerce')
    df['OG_APPTOPEN'] = pd.to_datetime(df['OG_APPTOPEN'], errors='coerce')
    early_cutoff = pd.Timedelta(hours=early_threshold_hours)

    # Column name reflects the threshold so the viewer knows what it means
    early_col = f'manual_cc_{early_threshold_hours}hr_before_og_apptopen'

    # Filter to manual CCs with a valid OG_APPTOPEN
    manual_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] != 'DATASCIENCE') &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1) &
        (df['OG_APPTOPEN'].notna())
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST', 'OG_APPTOPEN', 'UPDATE_USER',
       'EAR2NAME', 'EAR2ID', 'CUSTOMERCODE', 'ROLLUPBRANCHNAME']].copy()

    # Keep only CCs more than `early_threshold_hours` before OG_APPTOPEN
    very_early = manual_cc[
        (manual_cc['ENTERED_DATETIME_CST'] < manual_cc['OG_APPTOPEN']) &
        ((manual_cc['OG_APPTOPEN'] - manual_cc['ENTERED_DATETIME_CST']) > early_cutoff)
    ].copy()

    # --- Summary 1: By EAR2NAME ---
    by_ear2 = (
        very_early.groupby('EAR2NAME')
        .agg(
            distinct_loads=('LOAD_NUM', 'nunique'),
            **{early_col: ('ENTERED_DATETIME_CST', 'count')}
        )
        .reset_index()
        .sort_values(early_col, ascending=False)
    )

    # --- Summary 2: By UPDATE_USER + ROLLUPBRANCHNAME ---
    by_user_branch = (
        very_early.groupby(['UPDATE_USER', 'ROLLUPBRANCHNAME'])
        .agg(
            distinct_loads=('LOAD_NUM', 'nunique'),
            **{early_col: ('ENTERED_DATETIME_CST', 'count')}
        )
        .reset_index()
        .sort_values(early_col, ascending=False)
    )

    # --- Summary 3: By UPDATE_USER + ROLLUPBRANCHNAME + EAR2NAME ---
    by_user_branch_ear2 = (
        very_early.groupby(['UPDATE_USER', 'ROLLUPBRANCHNAME', 'EAR2NAME'])
        .agg(
            distinct_loads=('LOAD_NUM', 'nunique'),
            **{early_col: ('ENTERED_DATETIME_CST', 'count')}
        )
        .reset_index()
        .sort_values(early_col, ascending=False)
    )

    # --- Detail: Row-level dump of all qualifying manual CCs ---
    detail = (
        very_early[['LOAD_NUM', 'ENTERED_DATETIME_CST', 'UPDATE_USER',
                    'CUSTOMERCODE', 'EAR2NAME', 'EAR2ID']]
        .sort_values(['LOAD_NUM', 'ENTERED_DATETIME_CST'])
        .reset_index(drop=True)
    )

    return by_ear2, by_user_branch, by_user_branch_ear2, detail


# Run the function
early_by_ear2, early_by_user_branch, early_by_user_branch_ear2, early_detail = get_early_manual_cc_summary(df, early_threshold_hours=4)

print("=== Manual CCs >4 hrs before OG_APPTOPEN by EAR2NAME ===")
print(f"Distinct EAR2s with early touches: {len(early_by_ear2):,}")
print()
display(early_by_ear2.head(15))

print("\n=== Manual CCs >4 hrs before OG_APPTOPEN by UPDATE_USER / ROLLUPBRANCHNAME ===")
print(f"Distinct user/branch combos with early touches: {len(early_by_user_branch):,}")
print()
display(early_by_user_branch.head(20))

print("\n=== Manual CCs >4 hrs before OG_APPTOPEN by UPDATE_USER / ROLLUPBRANCHNAME / EAR2NAME ===")
print(f"Distinct user/branch/EAR2 combos with early touches: {len(early_by_user_branch_ear2):,}")
print()
display(early_by_user_branch_ear2.head(20))

print("\n=== Raw Manual CC Detail (>4 hrs before OG_APPTOPEN) ===")
print(f"Total qualifying manual CC rows: {len(early_detail):,}")
print()
display(early_detail.head(20))


=== Manual CCs >4 hrs before OG_APPTOPEN by EAR2NAME ===
Distinct EAR2s with early touches: 1,045



,EAR2NAME,distinct_loads,manual_cc_4hr_before_og_apptopen
906,The Coca-Cola Company,305,333
929,The Quaker Oats Company,186,192
226,"Constellation Brands, Inc.",176,184
425,Harvest Hill Beverage Company,103,108
123,"Bluetriton Brands, Inc.",100,102
462,"INDEPENDENT PURCHASING COOPERATIVE, INC.",78,91
487,International Paper Company,77,89
778,Robinson Fresh,81,86
887,Target Corporation,71,74
42,Amazon Robotics LLC,65,71



=== Manual CCs >4 hrs before OG_APPTOPEN by UPDATE_USER / ROLLUPBRANCHNAME ===
Distinct user/branch combos with early touches: 434



,UPDATE_USER,ROLLUPBRANCHNAME,distinct_loads,manual_cc_4hr_before_og_apptopen
48,CAMPJOS,Memphis Capacity,343,409
366,SLAPKAR,Chicago Central Capacity,231,276
213,LADIJEN,Capacity Procurement - Lean,213,213
288,OLIVTAY,Chicago Central Capacity,211,212
176,HOVAGAV,Chicago Central Capacity,172,184
314,RADDCHR,Michigan Capacity,163,171
57,CASTAN,Dallas Capacity,165,169
271,MOORHARO,NAST After Hours,161,162
86,CUBIGLO,Chicago Central Capacity,130,147
361,SHIMGAB,Kansas City Capacity,114,120



=== Manual CCs >4 hrs before OG_APPTOPEN by UPDATE_USER / ROLLUPBRANCHNAME / EAR2NAME ===
Distinct user/branch/EAR2 combos with early touches: 4,376



,UPDATE_USER,ROLLUPBRANCHNAME,EAR2NAME,distinct_loads,manual_cc_4hr_before_og_apptopen
2616,MOORHARO,NAST After Hours,The Quaker Oats Company,100,100
561,CAMPJOS,Memphis Capacity,International Paper Company,48,57
538,CAMPJOS,Memphis Capacity,"Floor And Decor Outlets Of America, Inc.",45,51
3483,SALAADR,TC Capacity,Robinson Fresh,30,30
526,CAMPJOS,Memphis Capacity,"Constellation Brands, Inc.",20,26
986,CUBIGLO,Chicago Central Capacity,The Coca-Cola Company,17,21
2825,OLIVTAY,Chicago Central Capacity,The Coca-Cola Company,19,19
3728,SILVMIT,Chicago Central Capacity,Amazon Robotics LLC,15,19
9,AHMABIL,NAST After Hours,"Link Snacks, Inc.",18,18
607,CAMPJOS,Memphis Capacity,The Coca-Cola Company,16,17



=== Raw Manual CC Detail (>4 hrs before OG_APPTOPEN) ===
Total qualifying manual CC rows: 6,875



,LOAD_NUM,ENTERED_DATETIME_CST,UPDATE_USER,CUSTOMERCODE,EAR2NAME,EAR2ID
0,544142530,2026-05-07 13:22:00+00:00,CASTAN,C8142223,Waterloo Sparkling Water Corp.,788264.0
1,545140023,2026-05-22 15:17:00+00:00,PIELCHE,C2889348,Walmart Inc.,103.0
2,545173337,2026-05-14 12:03:00+00:00,PIEKJOS,C8012738,Ruxton Chocolates LLC,28865.0
3,545598199,2026-05-21 15:36:00+00:00,SLAPKAR,C955950,"Prairie Farms Dairy, Inc.",24138.0
4,545823196,2026-05-01 18:54:00+00:00,LADIJEN,C4439478,The Coca-Cola Company,350.0
5,546316972,2026-05-26 13:03:00+00:00,WHITRYAN,C3013029,"Fuji Component Parts USA, Inc.",427564.0
6,546334854,2026-05-05 04:05:00+00:00,CAMPJOS,C655645,"Floor And Decor Outlets Of America, Inc.",1025.0
7,546334854,2026-05-05 04:06:00+00:00,CAMPJOS,C655645,"Floor And Decor Outlets Of America, Inc.",1025.0
8,546334858,2026-05-05 04:06:00+00:00,CAMPJOS,C655645,"Floor And Decor Outlets Of America, Inc.",1025.0
9,546610486,2026-05-15 14:37:00+00:00,GRANHUN,C5166081,Omya Inc.,26222.0


In [ ]:
print("Total manual CCs >4 hrs before OG_APPTOPEN (EAR2 summary):                    ", early_by_ear2['manual_cc_4hr_before_og_apptopen'].sum())
print("Total manual CCs >4 hrs before OG_APPTOPEN (user/branch summary):            ", early_by_user_branch['manual_cc_4hr_before_og_apptopen'].sum())
print("Total manual CCs >4 hrs before OG_APPTOPEN (user/branch/EAR2 summary):      ", early_by_user_branch_ear2['manual_cc_4hr_before_og_apptopen'].sum())
print("Total manual CCs >4 hrs before OG_APPTOPEN (detail rows):                    ", len(early_detail))


Total manual CCs >4 hrs before OG_APPTOPEN (EAR2 summary):                     6862
Total manual CCs >4 hrs before OG_APPTOPEN (user/branch summary):             6875
Total manual CCs >4 hrs before OG_APPTOPEN (user/branch/EAR2 summary):       6862
Total manual CCs >4 hrs before OG_APPTOPEN (detail rows):                     6875


In [ ]:
def get_load_nums_for_user_early_cc(df, update_user, early_threshold_hours=4):
    """
    Returns a list of LOAD_NUMs where the specified UPDATE_USER entered at least
    one manual check call more than `early_threshold_hours` before OG_APPTOPEN.

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake.
        update_user (str): The UPDATE_USER value to filter on.
        early_threshold_hours (int): Hours before OG_APPTOPEN threshold. Default 4.

    Returns:
        list: Sorted list of distinct LOAD_NUMs meeting the criteria.
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], errors='coerce')
    df['OG_APPTOPEN'] = pd.to_datetime(df['OG_APPTOPEN'], errors='coerce')
    early_cutoff = pd.Timedelta(hours=early_threshold_hours)

    mask = (
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] == update_user) &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1) &
        (df['OG_APPTOPEN'].notna()) &
        (df['ENTERED_DATETIME_CST'] < df['OG_APPTOPEN']) &
        ((df['OG_APPTOPEN'] - df['ENTERED_DATETIME_CST']) > early_cutoff)
    )

    load_nums = sorted(df.loc[mask, 'LOAD_NUM'].unique().tolist())
    print(f"UPDATE_USER:          '{update_user}'")
    print(f"Threshold:            >{early_threshold_hours} hrs before OG_APPTOPEN")
    print(f"Qualifying LOAD_NUMs: {len(load_nums):,}")
    return load_nums


# Example usage — replace with the user you want to investigate
# example_user = early_by_user_branch.iloc[0]['UPDATE_USER']
example_user = 'CAMPJOS'
user_load_nums = get_load_nums_for_user_early_cc(df, update_user=example_user, early_threshold_hours=4)
print(f"\nFirst 20 LOAD_NUMs: {user_load_nums[:10]}")


UPDATE_USER:          'CAMPJOS'
Threshold:            >4 hrs before OG_APPTOPEN
Qualifying LOAD_NUMs: 343

First 20 LOAD_NUMs: [546334854, 546334858, 547652342, 548174299, 548176730, 548285578, 548285583, 548285587, 548285596, 548285604]


In [ ]:
# df[df['LOAD_NUM']==546334854][['LOAD_NUM','DESCRIPTION','ENTERED_DATETIME_CST','UPDATE_USER','OG_SCHEDDATETIME','OG_APPTOPEN']]

,LOAD_NUM,DESCRIPTION,ENTERED_DATETIME_CST,UPDATE_USER,OG_SCHEDDATETIME,OG_APPTOPEN
4263344,546334854,Check Call,2026-05-05 04:05:00+00:00,CAMPJOS,2026-04-28 17:47:00-05:00,2026-05-05 15:00:00-05:00
4263895,546334854,Check Call,2026-05-05 04:06:00+00:00,CAMPJOS,2026-04-28 17:47:00-05:00,2026-05-05 15:00:00-05:00
4876113,546334854,Checked In,2026-05-05 14:00:00+00:00,BookEvent,2026-04-28 17:47:00-05:00,2026-05-05 15:00:00-05:00
4910171,546334854,Arrived Pick,2026-05-05 14:26:00+00:00,EDI-214,2026-04-28 17:47:00-05:00,2026-05-05 15:00:00-05:00
4910180,546334854,Depart Pick,2026-05-05 14:26:00+00:00,EDI-214,2026-04-28 17:47:00-05:00,2026-05-05 15:00:00-05:00
4938277,546334854,Check Call,2026-05-05 14:46:00+00:00,EDI-214,2026-04-28 17:47:00-05:00,2026-05-05 15:00:00-05:00
4982424,546334854,Check Call,2026-05-05 15:16:00+00:00,EDI,2026-04-28 17:47:00-05:00,2026-05-05 15:00:00-05:00
5150272,546334854,Check Call,2026-05-05 17:14:00+00:00,EDI,2026-04-28 17:47:00-05:00,2026-05-05 15:00:00-05:00
5191314,546334854,Check Call,2026-05-05 17:44:00+00:00,EDI,2026-04-28 17:47:00-05:00,2026-05-05 15:00:00-05:00
5236325,546334854,Check Call,2026-05-05 18:15:00+00:00,EDI-214,2026-04-28 17:47:00-05:00,2026-05-05 15:00:00-05:00


In [ ]:
# # Export early manual CC summaries to Excel
# output_path = f'../data/processed/early_manual_cc_summary_{startdate}_to_{enddate}.xlsx'

# with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
#     early_by_ear2.to_excel(writer, sheet_name='By_EAR2NAME', index=False)
#     early_by_user_branch.to_excel(writer, sheet_name='By_User_Branch', index=False)
#     early_by_user_branch_ear2.to_excel(writer, sheet_name='By_User_Branch_EAR2', index=False)
#     early_detail.to_excel(writer, sheet_name='Detail', index=False)

# print(f"✓ Excel file saved to: {output_path}")
# print(f"  Tab 'By_EAR2NAME':           {len(early_by_ear2):,} rows")
# print(f"  Tab 'By_User_Branch':        {len(early_by_user_branch):,} rows")
# print(f"  Tab 'By_User_Branch_EAR2':   {len(early_by_user_branch_ear2):,} rows")
# print(f"  Tab 'Detail':                {len(early_detail):,} rows")


✓ Excel file saved to: ../data/processed/early_manual_cc_summary_2026-05-01_to_2026-05-31.xlsx
  Tab 'By_EAR2NAME':           1,045 rows
  Tab 'By_User_Branch':        434 rows
  Tab 'By_User_Branch_EAR2':   4,376 rows
  Tab 'Detail':                6,875 rows


In [ ]:
# loads_with_early_manual_cc[loads_with_early_manual_cc['LOAD_NUM']==554287901]

In [ ]:
print("Total Manual CCs before Pickup Open:", loads_with_early_manual_cc['manual_cc_before_pickup_count'].sum())

Total Manual CCs before Pickup Open: 15297


In [ ]:
def get_manual_cc_near_auto_in_transit(df, window_minutes=30):
    """
    Identifies per load how many manual check calls occurred during active transit
    (between first P-Open and last D-Close) AND were sandwiched between automated
    check calls — i.e., there is an auto CC within `window_minutes` BEFORE the
    manual CC AND an auto CC within `window_minutes` AFTER it.

    The intent is to measure over-processing: manual CCs entered by a dispatcher
    when automated tracking was already firing on schedule around the same time.

    Criteria for a manual check call:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == False
        - UPDATE_USER != 'DATASCIENCE'
        - HUMAN_ENTERED_CHECKCALL_FLAG == 1

    Criteria for an automated check call:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == True
        - HUMAN_ENTERED_CHECKCALL_FLAG == 0
        - UPDATE_USER == 'DATASCIENCE'

    Criteria for "sandwiched by auto CCs":
        - At least one auto CC on the same load within `window_minutes` BEFORE the manual CC
        - At least one auto CC on the same load within `window_minutes` AFTER the manual CC

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake.
        window_minutes (int): Proximity window in minutes. Default 60.

    Returns:
        pd.DataFrame: One row per load with:
            - LOAD_NUM
            - first_pickup_open: earliest P-Open datetime
            - last_dropoff_close: latest D-Close datetime
            - transit_manual_cc_total: total manual CCs in transit window
            - manual_cc_sandwiched_count: manual CCs with an auto CC before AND after within window
            - pct_sandwiched: percentage of transit manual CCs that were sandwiched
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], errors='coerce')
    window = pd.Timedelta(minutes=window_minutes)

    # --- Transit window boundaries per load ---
    first_pickup = (
        df[df['CHECK_CALL_TYPE'] == 'P-Open']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .min()
        .rename('first_pickup_open')
    )

    last_dropoff = (
        df[df['CHECK_CALL_TYPE'] == 'D-Close']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .max()
        .rename('last_dropoff_close')
    )

    transit_bounds = pd.concat([first_pickup, last_dropoff], axis=1).dropna().reset_index()

    # --- Manual CCs ---
    manual_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] != 'DATASCIENCE') &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1)
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST']].copy()

    # Keep only manual CCs inside the transit window
    manual_cc = manual_cc.merge(transit_bounds, on='LOAD_NUM', how='inner')
    manual_cc = manual_cc[
        (manual_cc['ENTERED_DATETIME_CST'] >= manual_cc['first_pickup_open']) &
        (manual_cc['ENTERED_DATETIME_CST'] <= manual_cc['last_dropoff_close'])
    ].copy()

    # --- Automated CCs: CC type, automated flag, no human entry, UPDATE_USER == 'DATASCIENCE' ---
    auto_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == True) &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 0) &
        (df['UPDATE_USER'] != 'DATASCIENCE')
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST']].rename(
        columns={'ENTERED_DATETIME_CST': 'auto_datetime'}
    ).copy()

    # --- Cross-join per load to check sandwich condition ---
    # signed_diff > 0: auto CC is AFTER the manual CC
    # signed_diff < 0: auto CC is BEFORE the manual CC
    merged = manual_cc.merge(auto_cc, on='LOAD_NUM', how='left')
    merged['signed_diff'] = merged['auto_datetime'] - merged['ENTERED_DATETIME_CST']

    merged['auto_before'] = (merged['signed_diff'] < pd.Timedelta(0)) & \
                            (merged['signed_diff'].abs() <= window)
    merged['auto_after']  = (merged['signed_diff'] > pd.Timedelta(0)) & \
                            (merged['signed_diff'].abs() <= window)

    # Per manual CC: require at least one auto before AND one auto after
    sandwich_flags = (
        merged.groupby(['LOAD_NUM', 'ENTERED_DATETIME_CST'])
        .agg(
            has_auto_before=('auto_before', 'any'),
            has_auto_after=('auto_after', 'any')
        )
        .reset_index()
    )
    sandwich_flags['sandwiched'] = sandwich_flags['has_auto_before'] & sandwich_flags['has_auto_after']

    manual_cc = manual_cc.merge(
        sandwich_flags[['LOAD_NUM', 'ENTERED_DATETIME_CST', 'sandwiched']],
        on=['LOAD_NUM', 'ENTERED_DATETIME_CST'],
        how='left'
    )
    manual_cc['sandwiched'] = manual_cc['sandwiched'].fillna(False)

    # --- Aggregate per load ---
    result = (
        manual_cc.groupby(['LOAD_NUM', 'first_pickup_open', 'last_dropoff_close'])
        .agg(
            transit_manual_cc_total=('ENTERED_DATETIME_CST', 'count'),
            manual_cc_sandwiched_count=('sandwiched', 'sum')
        )
        .reset_index()
    )

    result['manual_cc_sandwiched_count'] = result['manual_cc_sandwiched_count'].astype(int)
    result['pct_sandwiched'] = (
        result['manual_cc_sandwiched_count'] / result['transit_manual_cc_total'] * 100
    ).round(1)

    result = result.sort_values('manual_cc_sandwiched_count', ascending=False)

    return result


# Run the function
in_transit_overprocessing = get_manual_cc_near_auto_in_transit(df, window_minutes=30)
qualifying_loads = len(in_transit_overprocessing)
total_sandwiched = in_transit_overprocessing['manual_cc_sandwiched_count'].sum()
total_transit_manual = in_transit_overprocessing['transit_manual_cc_total'].sum()

print(f"Loads with manual CCs in transit window:           {qualifying_loads:,}")
print(f"Total manual CCs in transit window:                {total_transit_manual:,}")
print(f"Manual CCs sandwiched by auto CCs (±30 min):      {total_sandwiched:,}")
print(f"Overall % sandwiched:                              {total_sandwiched / total_transit_manual * 100:.1f}%")
in_transit_overprocessing.head(10)


Loads with manual CCs in transit window:           14,035
Total manual CCs in transit window:                18,714
Manual CCs sandwiched by auto CCs (±30 min):      8,354
Overall % sandwiched:                              44.6%


,LOAD_NUM,first_pickup_open,last_dropoff_close,transit_manual_cc_total,manual_cc_sandwiched_count,pct_sandwiched
787,551412317,2026-05-21 15:00:00+00:00,2026-05-27 16:00:00+00:00,19,19,100.0
10689,553916777,2026-05-21 15:00:00+00:00,2026-05-26 14:00:00+00:00,18,17,94.4
8793,553442682,2026-05-26 15:00:00+00:00,2026-06-01 12:30:00+00:00,20,17,85.0
470,551098226,2026-05-01 15:00:00+00:00,2026-05-04 16:00:00+00:00,16,16,100.0
1304,551733805,2026-05-01 15:00:00+00:00,2026-05-05 04:00:00+00:00,16,15,93.8
5742,552771412,2026-05-08 15:00:00+00:00,2026-05-12 22:00:00+00:00,15,15,100.0
1300,551733783,2026-05-22 15:00:00+00:00,2026-05-26 18:00:00+00:00,15,15,100.0
3998,552423490,2026-05-08 20:00:00+00:00,2026-05-12 03:00:00+00:00,14,14,100.0
13669,555098958,2026-05-29 14:00:00+00:00,2026-06-01 19:00:00+00:00,14,14,100.0
19,546748574,2026-05-15 15:00:00+00:00,2026-05-19 04:00:00+00:00,14,14,100.0


In [ ]:
def get_sandwiched_cc_summary(df, window_minutes=30):
    """
    Summarizes manual check calls that were sandwiched between automated CCs
    during active transit (between first P-Open and last D-Close) at three aggregated
    levels, plus a raw row-level detail table:
      1. EAR2NAME — to identify which customer segments drive the most sandwiched touches
      2. UPDATE_USER + ROLLUPBRANCHNAME — to identify which dispatcher/branch combos
         are entering the most redundant manual CCs alongside automation
      3. UPDATE_USER + ROLLUPBRANCHNAME + EAR2NAME — to identify which user/branch/customer
         segment combos are driving the most sandwiched manual CCs
      4. Raw detail — one row per qualifying sandwiched manual CC with load/user/customer context

    Uses the same sandwich logic as get_manual_cc_near_auto_in_transit:
        - Manual CC must fall between first P-Open and last D-Close
        - At least one auto CC within `window_minutes` BEFORE the manual CC
        - At least one auto CC within `window_minutes` AFTER the manual CC

    Criteria for a manual check call:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == False
        - UPDATE_USER != 'DATASCIENCE'
        - HUMAN_ENTERED_CHECKCALL_FLAG == 1

    Criteria for an automated check call:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == True
        - HUMAN_ENTERED_CHECKCALL_FLAG == 0
        - UPDATE_USER != 'DATASCIENCE'

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake.
        window_minutes (int): Proximity window in minutes. Default 30.

    Returns:
        tuple:
            - by_ear2 (pd.DataFrame): Aggregated by EAR2NAME
            - by_user_branch (pd.DataFrame): Aggregated by UPDATE_USER + ROLLUPBRANCHNAME
            - by_user_branch_ear2 (pd.DataFrame): Aggregated by UPDATE_USER + ROLLUPBRANCHNAME + EAR2NAME
            - detail (pd.DataFrame): Row-level dump of all qualifying sandwiched manual CCs with
                                     LOAD_NUM, ENTERED_DATETIME_CST, UPDATE_USER,
                                     CUSTOMERCODE, EAR2NAME, EAR2ID
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], errors='coerce')
    window = pd.Timedelta(minutes=window_minutes)

    # --- Transit window boundaries per load ---
    first_pickup = (
        df[df['CHECK_CALL_TYPE'] == 'P-Open']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .min()
        .rename('first_pickup_open')
    )
    last_dropoff = (
        df[df['CHECK_CALL_TYPE'] == 'D-Close']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .max()
        .rename('last_dropoff_close')
    )
    transit_bounds = pd.concat([first_pickup, last_dropoff], axis=1).dropna().reset_index()

    # --- Manual CCs with dimension columns ---
    manual_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] != 'DATASCIENCE') &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1)
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST', 'UPDATE_USER', 'EAR2NAME', 'EAR2ID',
       'CUSTOMERCODE', 'ROLLUPBRANCHNAME']].copy()

    # Keep only manual CCs inside the transit window
    manual_cc = manual_cc.merge(transit_bounds, on='LOAD_NUM', how='inner')
    manual_cc = manual_cc[
        (manual_cc['ENTERED_DATETIME_CST'] >= manual_cc['first_pickup_open']) &
        (manual_cc['ENTERED_DATETIME_CST'] <= manual_cc['last_dropoff_close'])
    ].copy()

    # --- Automated CCs ---
    auto_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == True) &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 0) &
        (df['UPDATE_USER'] != 'DATASCIENCE')
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST']].rename(
        columns={'ENTERED_DATETIME_CST': 'auto_datetime'}
    ).copy()

    # --- Sandwich detection ---
    merged = manual_cc.merge(auto_cc, on='LOAD_NUM', how='left')
    merged['signed_diff'] = merged['auto_datetime'] - merged['ENTERED_DATETIME_CST']
    merged['auto_before'] = (merged['signed_diff'] < pd.Timedelta(0)) & (merged['signed_diff'].abs() <= window)
    merged['auto_after']  = (merged['signed_diff'] > pd.Timedelta(0)) & (merged['signed_diff'].abs() <= window)

    sandwich_flags = (
        merged.groupby(['LOAD_NUM', 'ENTERED_DATETIME_CST'])
        .agg(has_auto_before=('auto_before', 'any'), has_auto_after=('auto_after', 'any'))
        .reset_index()
    )
    sandwich_flags['sandwiched'] = sandwich_flags['has_auto_before'] & sandwich_flags['has_auto_after']

    # Join sandwich flag back to manual CCs (preserving dimension columns)
    manual_cc = manual_cc.merge(
        sandwich_flags[['LOAD_NUM', 'ENTERED_DATETIME_CST', 'sandwiched']],
        on=['LOAD_NUM', 'ENTERED_DATETIME_CST'],
        how='left'
    )
    manual_cc['sandwiched'] = manual_cc['sandwiched'].fillna(False)

    # Keep only sandwiched rows for the summaries
    sandwiched = manual_cc[manual_cc['sandwiched']].copy()

    # --- Summary 1: By EAR2NAME ---
    by_ear2 = (
        sandwiched.groupby('EAR2NAME')
        .agg(
            distinct_loads=('LOAD_NUM', 'nunique'),
            manual_cc_sandwiched_count=('ENTERED_DATETIME_CST', 'count')
        )
        .reset_index()
        .sort_values('manual_cc_sandwiched_count', ascending=False)
    )

    # --- Summary 2: By UPDATE_USER + ROLLUPBRANCHNAME ---
    by_user_branch = (
        sandwiched.groupby(['UPDATE_USER', 'ROLLUPBRANCHNAME'])
        .agg(
            distinct_loads=('LOAD_NUM', 'nunique'),
            manual_cc_sandwiched_count=('ENTERED_DATETIME_CST', 'count')
        )
        .reset_index()
        .sort_values('manual_cc_sandwiched_count', ascending=False)
    )

    # --- Summary 3: By UPDATE_USER + ROLLUPBRANCHNAME + EAR2NAME ---
    by_user_branch_ear2 = (
        sandwiched.groupby(['UPDATE_USER', 'ROLLUPBRANCHNAME', 'EAR2NAME'])
        .agg(
            distinct_loads=('LOAD_NUM', 'nunique'),
            manual_cc_sandwiched_count=('ENTERED_DATETIME_CST', 'count')
        )
        .reset_index()
        .sort_values('manual_cc_sandwiched_count', ascending=False)
    )

    # --- Detail: Row-level dump of all qualifying sandwiched manual CCs ---
    detail = (
        sandwiched[['LOAD_NUM', 'ENTERED_DATETIME_CST', 'UPDATE_USER',
                    'CUSTOMERCODE', 'EAR2NAME', 'EAR2ID']]
        .sort_values(['LOAD_NUM', 'ENTERED_DATETIME_CST'])
        .reset_index(drop=True)
    )

    return by_ear2, by_user_branch, by_user_branch_ear2, detail


# Run the function
sandwiched_by_ear2, sandwiched_by_user_branch, sandwiched_by_user_branch_ear2, sandwiched_detail = get_sandwiched_cc_summary(df, window_minutes=30)

print("=== Manual CCs Sandwiched by Auto CCs (±30 min) by EAR2NAME ===")
print(f"Distinct EAR2s with sandwiched touches: {len(sandwiched_by_ear2):,}")
print()
display(sandwiched_by_ear2.head(15))

print("\n=== Manual CCs Sandwiched by Auto CCs (±30 min) by UPDATE_USER / ROLLUPBRANCHNAME ===")
print(f"Distinct user/branch combos with sandwiched touches: {len(sandwiched_by_user_branch):,}")
print()
display(sandwiched_by_user_branch.head(20))

print("\n=== Manual CCs Sandwiched by Auto CCs (±30 min) by UPDATE_USER / ROLLUPBRANCHNAME / EAR2NAME ===")
print(f"Distinct user/branch/EAR2 combos with sandwiched touches: {len(sandwiched_by_user_branch_ear2):,}")
print()
display(sandwiched_by_user_branch_ear2.head(20))

print("\n=== Raw Sandwiched Manual CC Detail ===")
print(f"Total qualifying sandwiched manual CC rows: {len(sandwiched_detail):,}")
print()
display(sandwiched_detail.head(20))


=== Manual CCs Sandwiched by Auto CCs (±30 min) by EAR2NAME ===
Distinct EAR2s with sandwiched touches: 1,396



,EAR2NAME,distinct_loads,manual_cc_sandwiched_count
467,"Fábrica de Jabón La Corona, S.A. de C.V.",101,730
165,BorgWarner Inc.,109,412
632,International Paper Company,322,392
379,Eaton Corporation,39,172
947,Phillips 66,76,156
1049,Robinson Fresh,126,136
1212,The Coca-Cola Company,113,119
822,Molson Coors Beverage Company,90,98
858,"NextEra Energy, Inc.",56,81
988,ProAmpac Holdings Inc.,69,77



=== Manual CCs Sandwiched by Auto CCs (±30 min) by UPDATE_USER / ROLLUPBRANCHNAME ===
Distinct user/branch combos with sandwiched touches: 575



,UPDATE_USER,ROLLUPBRANCHNAME,distinct_loads,manual_cc_sandwiched_count
86,CASTAN,Dallas Capacity,386,451
517,TOSCARM,NAST After Hours,78,189
387,OWENSTEP,NAST After Hours,165,169
157,FOSTHEA,NAST After Hours,138,161
526,TURNJASM,Memphis IP,89,137
2,AHMABIL,NAST After Hours,132,134
351,MOORHARO,NAST After Hours,79,134
394,PELARIC,NAST After Hours,26,131
500,STINBLAI,NAST After Hours,126,129
541,VAZQELI,NAST After Hours,51,120



=== Manual CCs Sandwiched by Auto CCs (±30 min) by UPDATE_USER / ROLLUPBRANCHNAME / EAR2NAME ===
Distinct user/branch/EAR2 combos with sandwiched touches: 4,690



,UPDATE_USER,ROLLUPBRANCHNAME,EAR2NAME,distinct_loads,manual_cc_sandwiched_count
4351,TOSCARM,NAST After Hours,"Fábrica de Jabón La Corona, S.A. de C.V.",77,185
4410,TURNJASM,Memphis IP,International Paper Company,89,137
3353,PELARIC,NAST After Hours,BorgWarner Inc.,19,110
4461,VAZQELI,NAST After Hours,"Fábrica de Jabón La Corona, S.A. de C.V.",46,109
1843,HERFER,NAST After Hours,"Fábrica de Jabón La Corona, S.A. de C.V.",49,100
3221,ORTEPEDR,NAST After Hours,"Fábrica de Jabón La Corona, S.A. de C.V.",54,98
3071,MOORHARO,NAST After Hours,Phillips 66,43,94
1091,CRUZMAN,NAST After Hours,BorgWarner Inc.,19,71
2680,LOPEPAT1,NAST After Hours,BorgWarner Inc.,27,70
212,ARELMAR,NAST After Hours,"Fábrica de Jabón La Corona, S.A. de C.V.",30,61



=== Raw Sandwiched Manual CC Detail ===
Total qualifying sandwiched manual CC rows: 8,354



,LOAD_NUM,ENTERED_DATETIME_CST,UPDATE_USER,CUSTOMERCODE,EAR2NAME,EAR2ID
0,540066491,2026-05-05 11:53:00+00:00,KAPLRUS,C8897795,Trader Joe's Company,67.0
1,543505728,2026-05-18 12:14:00+00:00,KUNIAND,C671020,CCI Manufacturing IL Corporation,297137.0
2,544683700,2026-05-01 17:27:00+00:00,CASTAN,C7348876,Generac Holdings Inc.,15639.0
3,544683742,2026-05-19 17:40:00+00:00,RODRTHO,C7348876,Generac Holdings Inc.,15639.0
4,544742538,2026-05-20 11:43:00+00:00,CLUMJOS,C8029570,"GC Ingredients, Inc",189139.0
5,545181989,2026-05-21 11:24:00+00:00,KALMROS,C7043477,"Hyland Filter Service, Inc.",445020.0
6,545861638,2026-05-05 20:22:00+00:00,DURKMIC,C2335955,"Shuford Yarns, LLC",104366.0
7,546010388,2026-05-27 17:13:00+00:00,SCHUJUS,C52818,"Ring Container Technologies, LLC",6508.0
8,546748574,2026-05-16 04:08:00+00:00,TOSCARM,C1320632,"Fábrica de Jabón La Corona, S.A. de C.V.",93218.0
9,546748574,2026-05-16 08:31:00+00:00,VAZQELI,C1320632,"Fábrica de Jabón La Corona, S.A. de C.V.",93218.0


In [ ]:
# # Export sandwiched CC summaries to Excel
# output_path = f'../data/processed/sandwiched_cc_summary_{startdate}_to_{enddate}.xlsx'

# with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
#     sandwiched_by_ear2.to_excel(writer, sheet_name='By_EAR2NAME', index=False)
#     sandwiched_by_user_branch.to_excel(writer, sheet_name='By_User_Branch', index=False)
#     sandwiched_by_user_branch_ear2.to_excel(writer, sheet_name='By_User_Branch_EAR2', index=False)
#     sandwiched_detail.to_excel(writer, sheet_name='Detail', index=False)

# print(f"✓ Excel file saved to: {output_path}")
# print(f"  Tab 'By_EAR2NAME':           {len(sandwiched_by_ear2):,} rows")
# print(f"  Tab 'By_User_Branch':        {len(sandwiched_by_user_branch):,} rows")
# print(f"  Tab 'By_User_Branch_EAR2':   {len(sandwiched_by_user_branch_ear2):,} rows")
# print(f"  Tab 'Detail':                {len(sandwiched_detail):,} rows")


✓ Excel file saved to: ../data/processed/sandwiched_cc_summary_2026-05-01_to_2026-05-31.xlsx
  Tab 'By_EAR2NAME':           1,396 rows
  Tab 'By_User_Branch':        575 rows
  Tab 'By_User_Branch_EAR2':   4,690 rows
  Tab 'Detail':                8,354 rows


In [ ]:
def get_loads_with_manual_cc_after_dropoff(df):
    """
    Identifies LOAD_NUMs that have at least one manual check call
    (CHECK_CALL_TYPE = 'CC' and AUTOMATED = False) recorded AFTER
    the last drop off close event (CHECK_CALL_TYPE = 'D-Close').
    Excludes check calls where UPDATE_USER = 'DATASCIENCE'.
    Only includes check calls where HUMAN_ENTERED_CHECKCALL_FLAG = 1.

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake containing columns:
                           LOAD_NUM, CHECK_CALL_TYPE, AUTOMATED, ENTERED_DATETIME_CST,
                           UPDATE_USER, HUMAN_ENTERED_CHECKCALL_FLAG

    Returns:
        pd.DataFrame: DataFrame of qualifying LOAD_NUMs with supporting details:
                      - LOAD_NUM
                      - last_dropoff_close: latest D-Close datetime
                      - manual_cc_after_dropoff_count: number of manual CCs after D-Close
                      - latest_manual_cc: datetime of the last manual CC after D-Close
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], errors='coerce')

    # Get the last D-Close datetime per load
    dropoff_close = (
        df[df['CHECK_CALL_TYPE'] == 'D-Close']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .max()
        .rename('last_dropoff_close')
        .reset_index()
    )

    # Get manual check calls (CC + not automated + not DATASCIENCE user + human entered)
    manual_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] != 'DATASCIENCE') &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1)
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST']].copy()

    # Join manual CCs with their load's last D-Close datetime
    manual_cc = manual_cc.merge(dropoff_close, on='LOAD_NUM', how='inner')

    # Keep only manual CCs that occurred AFTER the last D-Close
    manual_cc_after = manual_cc[
        manual_cc['ENTERED_DATETIME_CST'] > manual_cc['last_dropoff_close']
    ]

    # Aggregate per load
    result = (
        manual_cc_after.groupby(['LOAD_NUM', 'last_dropoff_close'])
        .agg(
            manual_cc_after_dropoff_count=('ENTERED_DATETIME_CST', 'count'),
            latest_manual_cc=('ENTERED_DATETIME_CST', 'max')
        )
        .reset_index()
        .sort_values('manual_cc_after_dropoff_count', ascending=False)
    )

    return result


# Run the function
loads_with_late_manual_cc = get_loads_with_manual_cc_after_dropoff(df)
qualifying_loads_after = len(loads_with_late_manual_cc)

print(f"Total distinct LOAD_NUMs in dataset:               {total_loads:,}")
print(f"LOAD_NUMs with manual CC after drop off close:     {qualifying_loads_after:,}")
print(f"Percentage:                                        {qualifying_loads_after / total_loads * 100:.1f}%")
print(f"Total Manual CCs after Drop Off Close:             {loads_with_late_manual_cc['manual_cc_after_dropoff_count'].sum():,}")
loads_with_late_manual_cc.head(10)


Total distinct LOAD_NUMs in dataset:               311,086
LOAD_NUMs with manual CC after drop off close:     898
Percentage:                                        0.3%
Total Manual CCs after Drop Off Close:             1,040


,LOAD_NUM,last_dropoff_close,manual_cc_after_dropoff_count,latest_manual_cc
763,554146525,2026-05-23 19:00:00+00:00,9,2026-05-25 12:30:00+00:00
645,553654474,2026-05-23 14:00:00+00:00,9,2026-05-23 19:31:00+00:00
412,552901255,2026-05-13 20:00:00+00:00,8,2026-05-15 11:18:00+00:00
215,552282398,2026-05-05 22:00:00+00:00,8,2026-05-06 12:35:00+00:00
825,554495801,2026-05-27 17:00:00+00:00,7,2026-05-29 11:25:00+00:00
394,552859642,2026-05-09 07:00:00+00:00,7,2026-05-10 14:58:00+00:00
148,552046963,2026-05-08 02:00:00+00:00,6,2026-05-08 13:08:00+00:00
173,552153035,2026-05-06 22:00:00+00:00,6,2026-05-07 13:11:00+00:00
781,554260994,2026-05-25 16:00:00+00:00,6,2026-05-28 09:37:00+00:00
188,552196199,2026-05-05 16:00:00+00:00,5,2026-05-06 13:18:00+00:00


In [ ]:
def get_manual_cc_after_last_auto(df):
    """
    Identifies per load how many manual check calls occurred AFTER the last
    automated check call AND BEFORE the last D-Close.

    These represent manual touches that were required because automated tracking
    stopped firing before the load was delivered — an opportunity to reduce
    over-processing by improving automation continuity.

    Criteria for a manual check call:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == False
        - UPDATE_USER != 'DATASCIENCE'
        - HUMAN_ENTERED_CHECKCALL_FLAG == 1

    Criteria for an automated check call:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == True
        - HUMAN_ENTERED_CHECKCALL_FLAG == 0
        - UPDATE_USER == 'DATASCIENCE'

    Window of interest per load:
        - Start: last automated CC datetime (exclusive)
        - End:   last D-Close datetime (inclusive)

    Only loads that have both an automated CC and a D-Close are included.
    Loads where the last auto CC is already after the last D-Close are excluded.

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake.

    Returns:
        pd.DataFrame: One row per qualifying load with:
            - LOAD_NUM
            - last_auto_cc: datetime of the last automated CC on the load
            - last_dropoff_close: datetime of the last D-Close on the load
            - auto_gap_minutes: minutes between last auto CC and last D-Close
            - manual_cc_in_gap_count: manual CCs in the gap (after last auto, before D-Close)
            - earliest_gap_manual_cc: datetime of the first manual CC in the gap
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], errors='coerce')

    # --- Last automated CC per load ---
    last_auto = (
        df[
            (df['CHECK_CALL_TYPE'] == 'CC') &
            (df['AUTOMATED'] == True) &
            (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 0) &
            (df['UPDATE_USER'] != 'DATASCIENCE')
        ]
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .max()
        .rename('last_auto_cc')
        .reset_index()
    )

    # --- Last D-Close per load ---
    last_dropoff = (
        df[df['CHECK_CALL_TYPE'] == 'D-Close']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .max()
        .rename('last_dropoff_close')
        .reset_index()
    )

    # --- Combine: only loads where last auto CC is BEFORE last D-Close ---
    bounds = last_auto.merge(last_dropoff, on='LOAD_NUM', how='inner')
    bounds = bounds[bounds['last_auto_cc'] < bounds['last_dropoff_close']].copy()
    bounds['auto_gap_minutes'] = (
        (bounds['last_dropoff_close'] - bounds['last_auto_cc'])
        .dt.total_seconds() / 60
    ).round(1)

    # --- Manual CCs ---
    manual_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] != 'DATASCIENCE') &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1)
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST']].copy()

    # Keep only manual CCs that fall strictly after the last auto CC and before/at D-Close
    manual_cc = manual_cc.merge(bounds, on='LOAD_NUM', how='inner')
    manual_cc_in_gap = manual_cc[
        (manual_cc['ENTERED_DATETIME_CST'] > manual_cc['last_auto_cc']) &
        (manual_cc['ENTERED_DATETIME_CST'] <= manual_cc['last_dropoff_close'])
    ]

    # --- Aggregate per load ---
    result = (
        manual_cc_in_gap.groupby(['LOAD_NUM', 'last_auto_cc', 'last_dropoff_close', 'auto_gap_minutes'])
        .agg(
            manual_cc_in_gap_count=('ENTERED_DATETIME_CST', 'count'),
            earliest_gap_manual_cc=('ENTERED_DATETIME_CST', 'min')
        )
        .reset_index()
        .sort_values('manual_cc_in_gap_count', ascending=False)
    )

    return result, bounds


# Run the function
manual_cc_after_last_auto, gap_bounds = get_manual_cc_after_last_auto(df)
loads_with_auto_gap = len(manual_cc_after_last_auto)
total_gap_manual = manual_cc_after_last_auto['manual_cc_in_gap_count'].sum()
avg_gap_minutes = gap_bounds['auto_gap_minutes'].median()

print(f"Total distinct LOAD_NUMs in dataset:               {total_loads:,}")
print(f"Loads where last auto CC precedes D-Close:         {len(gap_bounds):,}")
print(f"Loads with manual CCs filling the auto gap:        {loads_with_auto_gap:,}")
print(f"  % of gap loads with manual fill-in:              {loads_with_auto_gap / len(gap_bounds) * 100:.1f}%")
print(f"Total manual CCs in auto gap:                      {total_gap_manual:,}")
print(f"Median gap duration (last auto CC → D-Close):      {avg_gap_minutes:.0f} min")
manual_cc_after_last_auto.head(10)


Total distinct LOAD_NUMs in dataset:               311,086
Loads where last auto CC precedes D-Close:         182,309
Loads with manual CCs filling the auto gap:        993
  % of gap loads with manual fill-in:              0.5%
Total manual CCs in auto gap:                      1,259
Median gap duration (last auto CC → D-Close):      255 min


,LOAD_NUM,last_auto_cc,last_dropoff_close,auto_gap_minutes,manual_cc_in_gap_count,earliest_gap_manual_cc
303,552313222,2026-05-04 20:19:00+00:00,2026-05-07 23:00:00+00:00,4481.0,19,2026-05-04 23:31:00+00:00
941,554871037,2026-05-26 17:26:00+00:00,2026-05-28 23:00:00+00:00,3214.0,13,2026-05-26 23:45:00+00:00
601,553360137,2026-05-14 22:29:00+00:00,2026-05-15 22:00:00+00:00,1411.0,12,2026-05-15 00:21:00+00:00
321,552404766,2026-05-04 22:51:00+00:00,2026-05-06 16:00:00+00:00,2469.0,11,2026-05-05 03:30:00+00:00
262,552223612,2026-05-13 22:41:00+00:00,2026-05-18 15:30:00+00:00,6769.0,8,2026-05-14 04:53:00+00:00
880,554311761,2026-05-24 12:30:00+00:00,2026-05-26 15:00:00+00:00,3030.0,6,2026-05-24 19:47:00+00:00
356,552527278,2026-05-05 17:59:00+00:00,2026-05-06 21:00:00+00:00,1621.0,6,2026-05-05 23:58:00+00:00
23,550784132,2026-05-03 00:09:00+00:00,2026-05-04 17:00:00+00:00,2451.0,5,2026-05-03 02:06:00+00:00
320,552404168,2026-05-04 20:35:00+00:00,2026-05-08 20:00:00+00:00,5725.0,5,2026-05-05 16:15:00+00:00
894,554429980,2026-05-22 15:21:00+00:00,2026-05-23 00:00:00+00:00,519.0,4,2026-05-22 17:33:00+00:00


In [ ]:
df[(df['LOAD_NUM']==546748574) & (pd.isnull(df['JOB_FAMILY_DESCRIPTION'])==False)][['LOAD_NUM','EAR2NAME','DESCRIPTION','ENTERED_DATETIME_CST','UPDATE_USER']]

,LOAD_NUM,EAR2NAME,DESCRIPTION,ENTERED_DATETIME_CST,UPDATE_USER
21134466,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-16 04:08:00+00:00,TOSCARM
21370101,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-16 08:31:00+00:00,VAZQELI
21649854,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-16 13:45:00+00:00,VAZQELI
21951904,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-16 19:02:00+00:00,ALTAELI
22245420,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-17 00:26:00+00:00,TOSCARM
22452047,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-17 04:34:00+00:00,TOSCARM
22608208,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-17 07:56:00+00:00,SOTOALEJ
22750287,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-17 11:05:00+00:00,SOTOALEJ
23001671,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-17 16:20:00+00:00,VAZQELI
23211658,546748574,"Fábrica de Jabón La Corona, S.A. de C.V.",Check Call,2026-05-17 20:28:00+00:00,VAZQELI
